# 1. Project Title: SentimentScope - XGBoost Model Training

This notebook trains a XGBoost classifier on the balanced dataset and evaluates its performance.

## 2. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import sys
import joblib

sys.path.append("../src")
from text_normalizer import clean_text
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder


## 3. Load Dataset

In [2]:
CSV_PATH = "../data/balanced_reviews.csv"
MODEL_PATH = "../models/xgboost_model.pkl"
VECTORIZER_PATH = "../models/tfidf_vectorizer_xgboost.pkl"

print("Loading balanced dataset...")
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")

Loading balanced dataset...
Shape: (26400, 6)


## 4. Data Inspection & Cleaning

In [3]:
print("Missing values inspection:")
print(df[['review_text', 'sentiment']].isnull().sum())

print("\nCleaning missing values...")
df = df.dropna(subset=['review_text', 'sentiment'])
print(f"Shape after cleaning: {df.shape}")

print("\nPreprocessing text column...")
df['clean_review'] = df['review_text'].astype(str).apply(clean_text)
print("Sample cleaned review:")
print(df['clean_review'].iloc[0])

Missing values inspection:
review_text    0
sentiment      0
dtype: int64

Cleaning missing values...
Shape after cleaning: (26400, 6)

Preprocessing text column...
Sample cleaned review:
great product provide good support while heavy lifting


## 5. Train/Test Split (80/20)

In [4]:
print("Splitting dataset into train and test sets...")
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Train set size: {X_train.shape[0]} | Test set size: {X_test.shape[0]}")

Splitting dataset into train and test sets...
Train set size: 21120 | Test set size: 5280


## 6. TF-IDF Vectorization

In [5]:
print("Fitting TF-IDF Vectorizer with Unigrams + Bigrams...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=15000, stop_words="english")
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)
print(f"TF-IDF feature space shape: {X_train_vectorized.shape}")

Fitting TF-IDF Vectorizer with Unigrams + Bigrams...
TF-IDF feature space shape: (21120, 15000)


## 7. Model Training

In [6]:
print("Encoding labels for model training...")
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

try:
    import xgboost as xgb
    from xgboost import XGBClassifier
    print("XGBoost library found. Initializing XGBClassifier...")
    model = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, eval_metric='mlogloss')
    model.fit(X_train_vectorized, y_train_encoded)
    print("XGBoost model training complete!")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    print("XGBoost not available. Falling back to GradientBoostingClassifier (this will take longer)...")
    model = GradientBoostingClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_vectorized, y_train_encoded)
    print("GradientBoostingClassifier training complete!")


Encoding labels for model training...
XGBoost not available. Falling back to GradientBoostingClassifier (this will take longer)...
GradientBoostingClassifier training complete!


## 8. Prediction

In [7]:
print("Making predictions on the test set...")
y_pred_encoded = model.predict(X_test_vectorized)
y_pred = label_encoder.inverse_transform(y_pred_encoded)
print("Prediction complete.")


Making predictions on the test set...
Prediction complete.


## 9. Evaluation Metrics

In [8]:
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("==================================================")
print("EVALUATION METRICS")
print("==================================================")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

EVALUATION METRICS
Accuracy:  0.7553
Precision: 0.7623
Recall:    0.7553
F1 Score:  0.7542


## 10. Classification Report

In [9]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.82      0.65      0.73      1760
     Neutral       0.69      0.79      0.74      1760
    Positive       0.77      0.82      0.80      1760

    accuracy                           0.76      5280
   macro avg       0.76      0.76      0.75      5280
weighted avg       0.76      0.76      0.75      5280



## 11. Confusion Matrix

In [10]:
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred, labels=["Negative", "Neutral", "Positive"]))

Confusion Matrix:

[[1145  401  214]
 [ 156 1393  211]
 [  95  215 1450]]


## 12. Save Model (.pkl)

In [11]:
print(f"Saving trained model to: {MODEL_PATH}...")
joblib.dump(model, MODEL_PATH)
print("Model saved successfully!")

Saving trained model to: ../models/xgboost_model.pkl...
Model saved successfully!


## 13. Save TF-IDF Vectorizer (.pkl)

In [12]:
print(f"Saving TF-IDF vectorizer to: {VECTORIZER_PATH}...")
joblib.dump(vectorizer, VECTORIZER_PATH)
print("Vectorizer saved successfully!")

Saving TF-IDF vectorizer to: ../models/tfidf_vectorizer_xgboost.pkl...
Vectorizer saved successfully!
